# Trotter分解による時間発展（量子回路）

量子コンピュータで実行できるのは、1〜2量子ビットに作用する小さなユニタリ（ゲート）の積と測定だけである。時間発展$e^{-iHt}$を量子回路で実行するために、**Trotter分解**でゲートの積に近似する。

各模型への適用（ハミルトニアンの分け方、具体的な回路、その模型での誤差の特徴、手計算）は、`models/<模型>/03_trotter.ipynb` にまとめてある。

## 1. Trotter分解

ハミルトニアンが$H = \sum_{k} H_k$と、それぞれは扱いやすい項の和で書けているとする。各項が互いに交換しなければ$e^{-iHt}\neq\prod_k e^{-iH_kt}$であり、$e^{-iHt}$をそのまま簡単なゲートに分解することはできない。そこで次の公式（Lie-Trotter積公式）を使う。

\begin{equation}
e^{-i(A+B)t} = \lim_{r\to\infty}\left(e^{-iB\,\delta t}e^{-iA\,\delta t}\right)^r,\quad \delta t = \frac{t}{r}
\end{equation}

項が3つ以上でも同様に$e^{-iHt}\approx\left(\prod_k e^{-iH_k\delta t}\right)^r$とできる。これは量子回路とは関係ない数学の公式で、量子モンテカルロ法や偏微分方程式の数値解法などでも使われている。量子回路で役に立つのは、各$H_k$を**単独で**指数関数にしたものがゲートで書けるからである。

## 2. 指数関数をゲートにする

- **交換する項をまとめる**：互いに交換する項の和$H_k = \sum_a h_a P_a$（$P_a$はPauli演算子の積）なら、$e^{-iH_k\delta t} = \prod_a e^{-ih_aP_a\delta t}$とちょうど分解できる。そこで$H$を「中の項がすべて交換するグループ」に分け、グループごとに指数関数をとる。

- **Pauli演算子の積の指数関数**：$e^{-i\theta P}$は次のように回路にできる。
  1. $P$に含まれる$X$は$H$ゲート、$Y$は$S^\dagger$と$H$ゲートで$Z$に変換する（基底変換）
  2. 対象の量子ビットのパリティをCNOTで1つの量子ビットに集め、$R_Z(2\theta)$をかける
  3. 1と2の逆をかけて戻す

  特に1〜2量子ビットの場合はそのままゲートになる。

  \begin{equation}
  e^{-i\theta X} = R_X(2\theta),\quad e^{-i\theta Z} = R_Z(2\theta),\quad e^{-i\theta Z\otimes Z} = R_{ZZ}(2\theta)
  \end{equation}

  ここで$R_X(\theta) = e^{-i\frac{\theta}{2}X}$、$R_{ZZ}(\theta) = e^{-i\frac{\theta}{2}Z\otimes Z}$などはQiskitの標準ゲートである。

- **並列化**：重ならない量子ビットに作用するゲートは同時にかけられるので、局所的なハミルトニアンなら1ステップの回路の深さを$n$によらない定数にできる。

## 3. Trotter誤差

Trotter分解は$r$が有限だと近似になる。2つの指数関数の積はBaker-Campbell-Hausdorff（BCH）公式より

\begin{equation}
e^{-iB\,\delta t}e^{-iA\,\delta t} = \exp\left[-i\delta t\left(A + B - \frac{i\delta t}{2}[B, A] + O(\delta t^2)\right)\right]
\end{equation}

となるので、1ステップごとに$O(\delta t^2)$の誤差が出る。$r$ステップでは誤差が積み重なり

\begin{equation}
\left\| e^{-i(A+B)t} - \left(e^{-iB\delta t}e^{-iA\delta t}\right)^r \right\| \le \frac{t^2}{2r}\left\|[A, B]\right\| = O\left(\frac{t^2}{r}\right)
\end{equation}

となる（**1次のTrotter分解**）。誤差を$\epsilon$以下にするには$r\sim t^2/\epsilon$ステップ、つまり誤差の逆数に比例するゲート数が必要になる。

両側を半分ずつにした**2次のTrotter分解**（鈴木分解）

\begin{equation}
e^{-i(A+B)\delta t} \approx e^{-iB\delta t/2}\,e^{-iA\delta t}\,e^{-iB\delta t/2}
\end{equation}

では$[B,A]$の項が打ち消し合い、誤差は$O(t^3/r^2)$に改善する。1ステップあたりのゲート数はほとんど変わらない。

さらに高次の分解（鈴木の高次公式）も作れ、$2k$次の分解では誤差が$O(t^{2k+1}/r^{2k})$になる。ただし1ステップあたりのゲート数が増えるので、精度とゲート数のバランスで選ぶ。

## 4. 測定

量子コンピュータで直接測れるのは、各量子ビットの$Z$基底の測定結果（0か1）だけである。Pauli演算子の積$P$の期待値を測るには、2節と同じ基底変換で$P$の各成分を$Z$に直してから測定し、結果のビット$b_j$から$Z_j = 1-2b_j = \pm1$の積をとる。これを多数回（ショット数$N_{\mathrm{shot}}$回）繰り返して平均すれば期待値の推定値になり、統計誤差は$1/\sqrt{N_{\mathrm{shot}}}$程度である。シミュレータでは状態ベクトルから期待値を直接計算することもできる。

---
**前へ**: [02 Jordan-Wigner変換](02_jordan_wigner.ipynb)　｜　**次へ**: [04 VQE](04_vqe.ipynb)